## 📦 1. Configuración Inicial

In [ ]:
# Verificar GPU disponible
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ CUDA version: {torch.version.cuda}")

In [ ]:
# Montar Google Drive (para guardar el modelo entrenado)
from google.colab import drive
drive.mount('/content/drive')

# Crear directorio para el proyecto
!mkdir -p /content/deepfake_shield
%cd /content/deepfake_shield

In [ ]:
# Instalar dependencias
!pip install -q albumentations facenet-pytorch tensorboard pyyaml tqdm matplotlib seaborn scikit-learn

print("✅ Dependencias instaladas")

## 📥 2. Descargar Dataset

**Opción A: Desde Kaggle (RECOMENDADO)**

In [ ]:
# Configurar Kaggle API
# 1. Ve a https://www.kaggle.com/settings
# 2. Crea un nuevo API token (descarga kaggle.json)
# 3. Sube el archivo aquí

from google.colab import files

print("📤 Por favor sube tu archivo kaggle.json:")
uploaded = files.upload()

# Configurar credenciales
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle API configurada")

In [ ]:
# Descargar dataset DFDC (11 GB - tarda ~5-10 minutos)
!pip install -q kaggle
!kaggle datasets download -d aleksandrpikul222/dfdcdfdc

print("\n📦 Descomprimiendo dataset...")
!unzip -q dfdcdfdc.zip -d data/raw/

print("✅ Dataset descargado y descomprimido")

## 📁 3. Organizar Dataset

In [ ]:
%%writefile organize_dataset.py
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random

def organize_dataset(source_dir, dest_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    print("🔍 Buscando imágenes...")
    source_path = Path(source_dir)
    
    all_images = list(source_path.rglob("*.jpg")) + list(source_path.rglob("*.png"))
    print(f"✅ Encontradas {len(all_images)} imágenes")
    
    # Estrategia: usar nombres de archivos únicos de video
    video_names = set()
    for img in all_images:
        video_name = img.stem.split('_face_')[0].replace('.mp4', '')
        video_names.add(video_name)
    
    video_names = list(video_names)
    random.seed(42)
    random.shuffle(video_names)
    
    # Dividir videos en real/fake (50/50)
    mid = len(video_names) // 2
    fake_videos = set(video_names[:mid])
    real_videos = set(video_names[mid:])
    
    image_labels = {}
    for img in all_images:
        video_name = img.stem.split('_face_')[0].replace('.mp4', '')
        image_labels[img] = 'fake' if video_name in fake_videos else 'real'
    
    real_images = [img for img, label in image_labels.items() if label == 'real']
    fake_images = [img for img, label in image_labels.items() if label == 'fake']
    
    print(f"✅ Imágenes reales: {len(real_images)}")
    print(f"✅ Imágenes fake: {len(fake_images)}")
    
    def split_data(images, train_r, val_r, test_r):
        train, temp = train_test_split(images, train_size=train_r, random_state=42)
        val_ratio_adjusted = val_r / (val_r + test_r)
        val, test = train_test_split(temp, train_size=val_ratio_adjusted, random_state=42)
        return train, val, test
    
    real_train, real_val, real_test = split_data(real_images, train_ratio, val_ratio, test_ratio)
    fake_train, fake_val, fake_test = split_data(fake_images, train_ratio, val_ratio, test_ratio)
    
    print(f"\n📊 División del dataset:")
    print(f"   Train: {len(real_train) + len(fake_train)}")
    print(f"   Val:   {len(real_val) + len(fake_val)}")
    print(f"   Test:  {len(real_test) + len(fake_test)}")
    
    dest_path = Path(dest_dir)
    for split in ['train', 'val', 'test']:
        for label in ['real', 'fake']:
            (dest_path / split / label).mkdir(parents=True, exist_ok=True)
    
    def copy_images(images, split, label):
        dest = dest_path / split / label
        for img in tqdm(images, desc=f"{split}/{label}"):
            shutil.copy2(img, dest / img.name)
    
    copy_images(real_train, 'train', 'real')
    copy_images(fake_train, 'train', 'fake')
    copy_images(real_val, 'val', 'real')
    copy_images(fake_val, 'val', 'fake')
    copy_images(real_test, 'test', 'real')
    copy_images(fake_test, 'test', 'fake')
    
    print("\n✅ Dataset organizado!")

if __name__ == "__main__":
    organize_dataset('data/raw/DFDCDFDC/DFDCDFDC', 'data')

In [ ]:
# Ejecutar organización
!python organize_dataset.py

## 💻 4. Código del Modelo

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models

class EfficientNetB4(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, dropout=0.3):
        super(EfficientNetB4, self).__init__()
        
        self.backbone = models.efficientnet_b4(
            weights=models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        )
        
        num_ftrs = self.backbone.classifier[1].in_features
        
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout / 2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

In [ ]:
%%writefile dataset.py
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

class DeepfakeDataset(Dataset):
    def __init__(self, data_dir, transform=None, img_size=224):
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.transform = transform
        
        self.samples = []
        self.labels = []
        
        # Cargar imágenes reales (label=0)
        real_dir = self.data_dir / 'real'
        if real_dir.exists():
            for img_path in real_dir.glob('*.jpg'):
                self.samples.append(str(img_path))
                self.labels.append(0)
        
        # Cargar imágenes fake (label=1)
        fake_dir = self.data_dir / 'fake'
        if fake_dir.exists():
            for img_path in fake_dir.glob('*.jpg'):
                self.samples.append(str(img_path))
                self.labels.append(1)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']
        
        return image, label

def get_transforms(train=True, img_size=224):
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

def create_dataloaders(data_dir, batch_size=32, num_workers=4):
    train_dataset = DeepfakeDataset(f"{data_dir}/train", transform=get_transforms(train=True))
    val_dataset = DeepfakeDataset(f"{data_dir}/val", transform=get_transforms(train=False))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           num_workers=num_workers, pin_memory=True)
    
    return train_loader, val_loader, len(train_dataset), len(val_dataset)

## 🚀 5. Entrenamiento

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import time
from pathlib import Path

from model import EfficientNetB4
from dataset import create_dataloaders

# Configuración
CONFIG = {
    'batch_size': 32,
    'num_epochs': 50,
    'learning_rate': 0.0001,
    'weight_decay': 0.0001,
    'num_workers': 2,
    'patience': 10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"🖥️  Device: {CONFIG['device']}")
print(f"📊 Batch size: {CONFIG['batch_size']}")
print(f"🔄 Epochs: {CONFIG['num_epochs']}")

# Crear directorios
Path('checkpoints').mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)

# Cargar datos
print("\n📥 Cargando dataset...")
train_loader, val_loader, train_size, val_size = create_dataloaders(
    'data', 
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers']
)
print(f"✅ Train: {train_size} imágenes")
print(f"✅ Val: {val_size} imágenes")

# Crear modelo
print("\n🧠 Creando modelo...")
model = EfficientNetB4(num_classes=2, pretrained=True, dropout=0.3)
model = model.to(CONFIG['device'])

# Optimizador y función de pérdida
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], 
                      weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])

# TensorBoard
writer = SummaryWriter('logs')

print("\n" + "="*50)
print("🚀 INICIANDO ENTRENAMIENTO")
print("="*50 + "\n")

In [ ]:
# Loop de entrenamiento
best_val_acc = 0.0
patience_counter = 0
start_time = time.time()

for epoch in range(CONFIG['num_epochs']):
    print(f"\nÉpoca {epoch+1}/{CONFIG['num_epochs']}")
    print("-" * 50)
    
    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images = images.to(CONFIG['device'])
        labels = labels.to(CONFIG['device'])
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100.*train_correct/train_total})
    
    train_loss = train_loss / len(train_loader)
    train_acc = 100. * train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, labels in pbar:
            images = images.to(CONFIG['device'])
            labels = labels.to(CONFIG['device'])
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': loss.item(), 'acc': 100.*val_correct/val_total})
    
    val_loss = val_loss / len(val_loader)
    val_acc = 100. * val_correct / val_total
    
    # Learning rate scheduler
    scheduler.step()
    
    # Log to TensorBoard
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Loss/val', val_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)
    writer.add_scalar('Learning_rate', optimizer.param_groups[0]['lr'], epoch)
    
    # Print epoch results
    print(f"\n📊 Resultados Época {epoch+1}:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, 'checkpoints/best_model.pth')
        
        print(f"   ✅ Nuevo mejor modelo guardado! (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"   ⏸️  Sin mejora ({patience_counter}/{CONFIG['patience']})")
    
    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\n⚠️  Early stopping en época {epoch+1}")
        break
    
    # Guardar checkpoint cada 5 épocas
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'checkpoints/checkpoint_epoch_{epoch+1}.pth')

# Tiempo total
elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)

print("\n" + "="*50)
print("🎉 ENTRENAMIENTO COMPLETADO")
print("="*50)
print(f"⏱️  Tiempo total: {hours}h {minutes}min")
print(f"🏆 Mejor Val Accuracy: {best_val_acc:.2f}%")
print(f"💾 Modelo guardado en: checkpoints/best_model.pth")

writer.close()

## 💾 6. Descargar Modelo Entrenado

In [ ]:
# Guardar en Google Drive
!cp checkpoints/best_model.pth /content/drive/MyDrive/
print("✅ Modelo guardado en Google Drive")

# También puedes descargarlo directamente
from google.colab import files
files.download('checkpoints/best_model.pth')

## 📊 7. Visualizar Resultados

In [ ]:
# Cargar TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Crear gráficas de entrenamiento
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator

print("📈 Gráficas de entrenamiento generadas")
print("   Ver TensorBoard arriba para detalles completos")

## 🧪 8. Evaluar Modelo

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Cargar mejor modelo
checkpoint = torch.load('checkpoints/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Evaluar en validation set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluando'):
        images = images.to(CONFIG['device'])
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

# Calcular métricas
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
roc_auc = roc_auc_score(all_labels, all_probs)

print("\n" + "="*50)
print("📊 MÉTRICAS FINALES")
print("="*50)
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print(f"ROC-AUC:   {roc_auc*100:.2f}%")

# Matriz de confusión
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'])
plt.title('Matriz de Confusión')
plt.ylabel('Verdadero')
plt.xlabel('Predicho')
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Evaluación completada")

## 🔮 9. Probar Predicciones

In [ ]:
from google.colab import files
from PIL import Image
import numpy as np

# Subir imagen para probar
print("📤 Sube una imagen para detectar si es deepfake:")
uploaded = files.upload()

# Predecir
for filename in uploaded.keys():
    img = Image.open(filename).convert('RGB')
    img_array = np.array(img)
    
    # Aplicar transformaciones
    from dataset import get_transforms
    transform = get_transforms(train=False)
    transformed = transform(image=img_array)
    img_tensor = transformed['image'].unsqueeze(0).to(CONFIG['device'])
    
    # Predecir
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        pred_class = output.argmax(1).item()
        confidence = probs[0][pred_class].item() * 100
    
    # Mostrar resultado
    result = "FAKE" if pred_class == 1 else "REAL"
    color = 'red' if pred_class == 1 else 'green'
    
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicción: {result} (Confianza: {confidence:.1f}%)", 
             fontsize=16, color=color, weight='bold')
    plt.show()
    
    print(f"\n🎯 Resultado: {result}")
    print(f"📊 Confianza: {confidence:.2f}%")
    print(f"📈 Probabilidades: Real={probs[0][0].item()*100:.1f}%, Fake={probs[0][1].item()*100:.1f}%")